# Fine-tuning Pipeline (MACE-MH-1 OMAT)

This notebook uses multi-head tuning with 2 datasets:
    * DFT relaxation set constructed below
    * configs from OMAT head w/ energy + force labels from MACE-MH-1 predictions (to preserve original model behavior)
      replay source [06/23/26]: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mh_1

See https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html for more details.

## Conda Env Setup
In the CLI, run 
`path/mace-mh-1.yml`

In [24]:
from ase.io import iread, write
from pathlib import Path

In [25]:
def make_outcar_paths(folder_path):
    # return a list of paths to the OUTCAR of each VASP run

    outcar_paths = []
    folder = Path(folder_path)
    for f in folder.iterdir():
        file = f / "OUTCAR"
        if file.is_file():
            outcar_paths.append(str(file))
    return outcar_paths

In [26]:
# ===== add DFT runs here =====

# OUTCARP_PATHS: list of paths to OUTCAR file for each run
OUTCAR_PATHS = make_outcar_paths("data/dft-data/")

In [27]:
print(OUTCAR_PATHS)

['data/dft-data/SCH3Good/OUTCAR', 'data/dft-data/OHTopBentSCH3Good/OUTCAR', 'data/dft-data/SCH3/OUTCAR']


In [32]:
# create finetune set by storing the first and last frames of each DFT run as an ASE Atoms object

def load_frames(path):
    # return tuple of first and last frames of DFT run
    
    frames = iread(path, index=":")
    next(frames)  # skip index 0, just geometry with no calc

    first_raw = next(frames)
    first = first_raw.copy()
    first.calc = first_raw.calc

    last = None
    for atoms in frames:
        last = atoms

    # first = next(iread(path, index="0"))
    # last = next(iread(path, index="-1"))

    return first, last

In [34]:
dft_frames_set = []
HEAD_NAME = "dft_ag"

for path in OUTCAR_PATHS:
    first, last = load_frames(path)
    for atoms in [first, last]:
        # atoms = atoms.get_potential_energy()
        atoms.info["REF_energy"] = atoms.calc.results["energy"]
        atoms.info["head"] = HEAD_NAME
        atoms.arrays["REF_forces"] = atoms.calc.results["forces"]
    dft_frames_set.extend([first, last])

write("data/mace-ft-input.xyz", dft_frames_set, format="extxyz")

**Create the combined finetuning set** with the following CLI command.
Run in the finetuned-mlips/data/ folder.
This calls fine_tuning_select.py (stored in the mace-torch package you installed in your conda env) 
to create a filtered 3rd file from your dft data and the replay. Edit flags below filepaths at will.

```bash 
python -m mace.cli.fine_tuning_select \
  --configs_pt data/replay-data-mh-1-omat-pbe.xyz \
  --configs_ft data/train.xyz \
  --num_samples 10000 \
  --subselect fps \
  --model path/to/foundation_model.model \
  --output selected_configs.xyz \
  --filtering_type combinations \
  --head_pt omat_pbe \
  --head_ft tuned_head \
  --weight_pt 1.0 \
  --weight_ft 10.0
```
Parameters:
    * `--configs_pt`: Path to the replay dataset
    * `--configs_ft`: Path to your target dataset
    * `--num_samples`: Number of configurations to select from the replay dataset
    * `--subselect`: Method for subselection (fps for Farthest Point Sampling or random)
    * `--filtering_type`: How to filter configurations based on elements:
        * `combinations`: Keep configurations with combinations of elements in your target dataset
        * `exclusive`: Keep configurations containing only elements in your target dataset
        * `inclusive`: Keep configurations containing all elements in your target dataset
        * `none`: No filtering
    * `--atomic_numbers`: Optionally specify specific atomic numbers to filter by
    * `--weight_pt`: weights the loss calculated on replay (pretraining refs) structures
    * `--weight_ft`: weights the loss calcualted on finetune structures

Copy/Paste Version:
```bash
python -m mace.cli.fine_tuning_select --configs_pt data/replay-data-mh-1-omat-pbe.xyz --configs_ft data/mace-ft-input.xyz --num_samples 10 --subselect random --model /Users/zschwab/.cache/huggingface/hub/models--mace-foundations--mace-mh-1/snapshots/95fc198fe533fd351b4b74bd0fdbdc6c10a2dddd/mace-mh-1.model --output selected_configs.xyz --filtering_type combinations --head_pt omat_pbe --head_ft tuned_head --weight_pt 1.0 --weight_ft 10.0
```

**Train model on combined set** with the following CLI command. (EDIT)

```bash
python -m mace.cli.run_train \
  --name myMACE_finetuned \
  --pt_train_file selected_configs.xyz \
  --train_file data/mace-ft-input.xyz \
  --valid_fraction 0.1 \
  --foundation_model path/to/foundation_model.model \
  --energy_weight 1.0 \
  --forces_weight 100.0 \
  --swa \
  --swa_energy_weight 10.0 \
  --swa_forces_weight 100.0
```
Parameters:
    * `--valid_fraction`: fraction of training data set aside for validation set (from mace-ft-input.xyz)

Successful version:
```bash
python -m mace.cli.run_train --name myMACE_finetuned_test --train_file data/mace-ft-input.xyz --foundation_model mh-1 --foundation_head omat_pbe --pt_train_file selected_configs.xyz --energy_weight 1.0 --forces_weight 100.0 --multiheads_finetuning True --valid_fraction 0.4 --E0s estimated --max_num_epochs 50
```